<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:


import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login


TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"


FEAT_START, FEAT_END   = "2026-01-01", "2026-02-28"
MID                    = "2026-01-31"
LABEL_START, LABEL_END = "2026-03-01", "2026-04-30"
MIN_IMPR = 100


con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month IN ('2026-01','2026-02')
  AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{FEAT_START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
""")
print("pages:", con.execute("SELECT COUNT(*) FROM content_slice").fetchone()[0])


con.execute(f"""
CREATE OR REPLACE TEMP TABLE dataset AS
WITH daily AS (
    SELECT d.client_hash_id, d.content_hash_id, d.report_date,
           d.gsc_impressions, d.gsc_clicks, d.gsc_sum_position,
           d.ga4_pageviews, d.ga4_sessions, d.ga4_users,
           d.ga4_engaged_sessions, d.ga4_total_engagement_sec,
           COALESCE(d.report_date >= CAST(c.ga4_data_start AS DATE), FALSE) AS ga4_ok
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN content_slice s
      ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
    JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
    WHERE d.month IN ('2026-01','2026-02','2026-03','2026-04')
      AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
      AND d.report_date >= CAST(c.gsc_data_start AS DATE)
),
agg AS (
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                         AS f01,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)                 AS f02,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)           AS f05,
      SUM(CASE WHEN ga4_ok THEN ga4_engaged_sessions END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f06,
      SUM(CASE WHEN ga4_ok THEN ga4_total_engagement_sec END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f07,
      SUM(CASE WHEN ga4_ok THEN ga4_pageviews END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f08,
      SUM(CASE WHEN ga4_ok THEN ga4_users END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f09
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
),
halves AS (
    SELECT client_hash_id, content_hash_id,
      (report_date > DATE '{MID}') AS is_h2,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                AS avg_impr,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)        AS ctr,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS pos
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) > 0
),
trend AS (
    SELECT client_hash_id, content_hash_id,
      (MAX(CASE WHEN is_h2 THEN avg_impr END) - MAX(CASE WHEN NOT is_h2 THEN avg_impr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN avg_impr END), 0)       AS f03,
      (MAX(CASE WHEN is_h2 THEN ctr END) - MAX(CASE WHEN NOT is_h2 THEN ctr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN ctr END), 0)            AS f04,
      MAX(CASE WHEN is_h2 THEN pos END) - MAX(CASE WHEN NOT is_h2 THEN pos END) AS f14
    FROM halves
    GROUP BY 1, 2
),
dim AS (
    SELECT content_hash_id,
      DATE_DIFF('day',
        CAST(CASE WHEN content_updated_date IS NOT NULL
                   AND content_updated_date <= DATE '{FEAT_END}'
                  THEN content_updated_date ELSE content_created_date END AS DATE),
        DATE '{FEAT_END}')                                                   AS f10,
      CASE WHEN keyword_hash_id IS NULL OR keyword_hash_id = '' THEN NULL
           ELSE COUNT(*) OVER (PARTITION BY client_hash_id, keyword_hash_id) - 1
      END                                                                    AS f11,
      CASE WHEN main_intent IS NULL THEN NULL
           ELSE DENSE_RANK() OVER (ORDER BY main_intent) - 1 END             AS f12,
      backlinks                                                              AS f13
    FROM read_parquet('{SRC_CONTENT}')
),
lab AS (
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN report_date END), 0)                                     AS avg_base,
      COALESCE(SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN report_date END), 0), 0)                                 AS avg_next
    FROM daily
    GROUP BY 1, 2
)
SELECT s.content_hash_id AS content_id,
       s.client_hash_id  AS client_id,
       a.f01 AS avg_daily_impressions,
       a.f02 AS ctr,
       t.f03 AS impressions_trend,
       t.f04 AS ctr_trend,
       a.f05 AS weighted_avg_position,
       a.f06 AS engagement_rate,
       a.f07 AS engagement_time_per_session,
       a.f08 AS avg_daily_pageviews,
       a.f09 AS avg_daily_users,
       m.f10 AS page_age_days,
       m.f11 AS same_keyword_pages,
       m.f12 AS main_intent_code,
       m.f13 AS backlinks,
       t.f14 AS position_trend,
       CASE WHEN l.avg_next < 0.8 * l.avg_base THEN TRUE
            ELSE FALSE END AS is_declining
FROM content_slice s
LEFT JOIN agg   a ON a.client_hash_id = s.client_hash_id AND a.content_hash_id = s.content_hash_id
LEFT JOIN trend t ON t.client_hash_id = s.client_hash_id AND t.content_hash_id = s.content_hash_id
LEFT JOIN lab   l ON l.client_hash_id = s.client_hash_id AND l.content_hash_id = s.content_hash_id
LEFT JOIN dim   m ON m.content_hash_id = s.content_hash_id
""")

df = con.execute("SELECT * FROM dataset").df()
print(df.shape)
df


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 82612


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(82612, 17)


,content_id,client_id,avg_daily_impressions,ctr,impressions_trend,ctr_trend,weighted_avg_position,engagement_rate,engagement_time_per_session,avg_daily_pageviews,avg_daily_users,page_age_days,same_keyword_pages,main_intent_code,backlinks,position_trend,is_declining
0,content_2beea676d1f82ac0,client_73cda7b4e4f265ea,3.237288,0.000000,1.737662,NaN,10.753927,NaN,NaN,NaN,NaN,3,0,0,0,-4.099198,False
1,content_6692551c97224ae7,client_ff644d8251367cbb,8.627119,0.003929,0.408642,NaN,10.170923,0.0,1.0,0.067797,0.067797,393,0,0,<NA>,-2.764364,False
2,content_37447482347acea2,client_ff644d8251367cbb,2.016949,0.000000,2.960165,NaN,10.789916,0.0,0.0,0.016949,0.016949,3,0,0,<NA>,-6.174524,False
3,content_d1e74562ab1359b3,client_ff644d8251367cbb,3.576271,0.000000,3.454932,NaN,14.246445,NaN,NaN,0.000000,0.000000,359,0,0,<NA>,5.539589,False
4,content_8e96590ddf859673,client_ff644d8251367cbb,26.915254,0.001259,1.723234,-1.000000,6.899874,0.0,0.0,0.067797,0.067797,401,0,0,<NA>,-0.346147,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82607,content_0a3dbc35519aaf88,client_d211cb07b9059bab,13.711864,0.000000,0.120913,NaN,9.386897,NaN,NaN,0.000000,0.000000,3,<NA>,<NA>,<NA>,0.007579,False
82608,content_98154a3b5baab58e,client_d211cb07b9059bab,3.338983,0.010152,NaN,NaN,7.664975,NaN,NaN,0.000000,0.000000,173,<NA>,<NA>,<NA>,NaN,False
82609,content_81028f7bce06c3ef,client_d211cb07b9059bab,12.932203,0.001311,0.015345,-1.000000,10.483617,NaN,NaN,0.000000,0.000000,3,<NA>,<NA>,<NA>,0.354423,False
82610,content_92b90353e195ddb4,client_d211cb07b9059bab,2.440678,0.013889,0.345604,NaN,8.944444,0.0,0.0,0.200000,0.100000,3,<NA>,<NA>,<NA>,-1.082765,True


In [2]:

import numpy as np

TEST_PAGES       = 0.20
PAGES_TOL        = 0.02
MIN_SHARE        = 0.25
MIN_TEST_CLIENTS = 5
MAX_TRIES        = 2000


def split_by_client(df, label_col="is_declining", group_col="client_id",
                    test_pages=TEST_PAGES, tol=PAGES_TOL,
                    min_share=MIN_SHARE, max_tries=MAX_TRIES,
                    min_test_clients=MIN_TEST_CLIENTS, start_seed=0):
    clients = np.sort(df[group_col].unique())
    classes = df[label_col].unique()
    sizes = df[group_col].value_counts()

    for seed in range(start_seed, max_tries):
        rng = np.random.default_rng(seed)
        shuffled = rng.permutation(clients)


        target = test_pages * len(df)
        test_clients, total = [], 0
        for c in shuffled:
            if total >= target:
                break
            test_clients.append(c)
            total += sizes[c]

        is_test = df[group_col].isin(test_clients)
        train, test = df[~is_test], df[is_test]

        if abs(len(test) / len(df) - test_pages) > tol:
            continue
        if test[group_col].nunique() < min_test_clients:
            continue

        tr_share = train[label_col].value_counts(normalize=True)
        te_share = test[label_col].value_counts(normalize=True)

        if all(tr_share.get(c, 0) >= min_share and te_share.get(c, 0) >= min_share
               for c in classes):
            return train.copy(), test.copy(), seed

    raise RuntimeError(f"No valid split in {max_tries} seeds")


train_df, test_df, seed = split_by_client(df, start_seed=0)

print("seed:", seed)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["is_declining"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["is_declining"].value_counts(normalize=True).round(3))

assert set(train_df["client_id"]).isdisjoint(test_df["client_id"])

seed: 25
train: 65636 pages | 23 clients
test:  16976 pages | 5 clients

train label share:
 is_declining
False    0.688
True     0.312
Name: proportion, dtype: float64

test label share:
 is_declining
False    0.58
True     0.42
Name: proportion, dtype: float64


In [3]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES = [
    "avg_daily_impressions", "ctr", "impressions_trend", "ctr_trend",
    "weighted_avg_position", "engagement_rate", "engagement_time_per_session",
    "avg_daily_pageviews", "avg_daily_users", "page_age_days",
    "same_keyword_pages", "main_intent_code", "backlinks", "position_trend",
]



X_train, y_train = train_df[FEATURES], train_df["is_declining"]
X_test,  y_test_grouped  = test_df[FEATURES],  test_df["is_declining"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred_grouped = rf.predict(X_test)
prob_grouped = rf.predict_proba(X_test)[:, 1]
print("test accuracy:", round(accuracy_score(y_test_grouped, pred_grouped), 4))
from sklearn.metrics import f1_score
print("macro F1:", round(f1_score(y_test_grouped, pred_grouped, average="macro"), 4))


KeyboardInterrupt: 

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Building the score

​```
score = 0.7 × minmax(P(decline)) + 0.3 × minmax(log10(avg_daily_impressions))
​```

The score ranks pages by how much a refresh is worth. It depends on:

- **Model probability:** how likely the page is to decline, from the Random Forest.
- **Page size and traffic:** how much visibility the page has, measured by average daily impressions. We apply `log10` so very large pages don't dominate the ranking.

Both parts are min-max scaled to 0-1 so the 70/30 weights reflect their real influence.




In [ ]:

import numpy as np
import pandas as pd

def minmax(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min())

p_norm = minmax(prob_grouped)
imp_norm = minmax(np.log10(test_df["avg_daily_impressions"]))

score = pd.DataFrame({
    "client_id": test_df["client_id"].values,
    "content_id": test_df["content_id"].values,
    "score": 0.7 * p_norm + 0.3 * imp_norm,
}).sort_values("score", ascending=False).reset_index(drop=True)
score


In [16]:
features = con.execute(f"""
WITH agg AS (
  SELECT d.client_hash_id, d.content_hash_id,
         SUM(d.gsc_impressions) AS impressions,
         SUM(d.gsc_clicks) AS clicks,
         SUM(d.gsc_sum_position) * 1.0 / NULLIF(SUM(d.gsc_impressions), 0) AS weighted_avg_position,
         SUM(d.ga4_sessions) AS sessions,
         SUM(d.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(d.ga4_sessions), 0) AS engagement_rate
  FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
  JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
  WHERE d.month IN ('2026-01','2026-02')
    AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    AND CAST(c.gsc_data_start AS DATE) < DATE '{FEAT_START}'
  GROUP BY d.client_hash_id, d.content_hash_id
  HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
),
tiered AS (
  SELECT *,
         AVG(CASE WHEN sessions >= 30 THEN engagement_rate END) OVER () AS avg_engagement_rate,
         CASE WHEN weighted_avg_position <= 3  THEN 'top_3'
              WHEN weighted_avg_position <= 10 THEN 'page_1'
              WHEN weighted_avg_position <= 20 THEN 'striking'
              ELSE 'deep' END AS tier
  FROM agg
),
tier_stats AS (
  SELECT tier, SUM(clicks) AS tier_clicks, SUM(impressions) AS tier_impr,
         MEDIAN(impressions) AS tier_median_impr
  FROM tiered GROUP BY tier
)
SELECT t.client_hash_id, t.content_hash_id, t.tier,
       t.weighted_avg_position,
       (t.clicks * 1.0 / t.impressions)
         / NULLIF((s.tier_clicks - t.clicks) * 1.0 / NULLIF(s.tier_impr - t.impressions, 0), 0)
         AS ctr_per_peers,
       CASE WHEN t.sessions >= 30
            THEN t.engagement_rate / NULLIF(t.avg_engagement_rate, 0) END AS engagement_vs_avg,
       t.impressions * 1.0 / NULLIF(s.tier_median_impr, 0) AS impressions_per_peers
FROM tiered t JOIN tier_stats s USING (tier)
""").df()
feat_test = test_df[["client_id", "content_id"]].merge(
    features,
    left_on=["client_id", "content_id"],
    right_on=["client_hash_id", "content_hash_id"],
    how="left",
).drop(columns=["client_hash_id", "content_hash_id"])

feat_test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_id,content_id,tier,weighted_avg_position,ctr_per_peers,engagement_vs_avg,impressions_per_peers
0,client_861cdcccf8049915,content_16f329e85e7420a2,deep,21.293814,3.057984,NaN,0.488665
1,client_861cdcccf8049915,content_7a0ea2f6f51b19ac,striking,17.333333,0.000000,NaN,0.348416
2,client_861cdcccf8049915,content_2f9ad208fa86aefe,deep,20.858896,0.000000,NaN,0.821159
3,client_861cdcccf8049915,content_8bcfe2a94c0fa487,striking,14.593516,0.000000,NaN,0.604827
4,client_861cdcccf8049915,content_34de6c740920b340,page_1,9.672420,0.698266,NaN,1.821803
...,...,...,...,...,...,...,...
16971,client_23a62021009f63c4,content_d903bd9d8c1551c5,deep,28.685345,0.000000,NaN,0.584383
16972,client_23a62021009f63c4,content_03804d6fa931d1b7,deep,32.862378,2.662495,NaN,3.367758
16973,client_23a62021009f63c4,content_09d67d44d0790ed9,striking,14.699894,1.611161,0.861228,15.660633
16974,client_23a62021009f63c4,content_0868c1cfb576c6aa,striking,10.438757,3.045926,2.598446,9.223228


,client_id,content_id,tier,weighted_avg_position,ctr_per_peers,engagement_rate,sessions,impressions_per_peers
0,client_861cdcccf8049915,content_16f329e85e7420a2,deep,21.293814,3.057326,NaN,NaN,inf
1,client_861cdcccf8049915,content_7a0ea2f6f51b19ac,striking,17.333333,0.000000,NaN,NaN,1.241935
2,client_861cdcccf8049915,content_2f9ad208fa86aefe,deep,20.858896,0.000000,NaN,NaN,inf
3,client_861cdcccf8049915,content_8bcfe2a94c0fa487,striking,14.593516,0.000000,NaN,NaN,2.155914
4,client_861cdcccf8049915,content_34de6c740920b340,page_1,9.672420,0.695904,NaN,NaN,8.777778
...,...,...,...,...,...,...,...,...
16971,client_23a62021009f63c4,content_d903bd9d8c1551c5,deep,28.685345,0.000000,0.000000,5.0,inf
16972,client_23a62021009f63c4,content_03804d6fa931d1b7,deep,32.862378,2.661919,0.090909,11.0,inf
16973,client_23a62021009f63c4,content_09d67d44d0790ed9,striking,14.699894,1.608978,0.052083,96.0,55.822581
16974,client_23a62021009f63c4,content_0868c1cfb576c6aa,striking,10.438757,3.041793,0.157143,70.0,32.876344


In [9]:
print(test_df["weighted_avg_position"].isna().sum(), "من", len(test_df))
print(feat_test["weighted_avg_position"].isna().sum(), "من", len(feat_test))

0 من 16976
0 من 16976


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.